# Preparation, fitting and residual evaluation

Fictional dimensionless observations; one nonfinite row is explicitly excluded. Ordinary scientific calls span cells. Retained native references link stages. An independent closed-form OLS oracle checks parameters, covariance and residuals.

The final cell exercises actual optimizer failure/retry, hard exit, minimal capture and intermediate-file loss/alteration/restoration. This is a controlled Lab scenario, not automatic capture or replay. Tracked by uibcdf/recorda-lab#10.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("experiments").resolve()))
import recorda
from inspect_workflow import inspect_workflow
from run_workflow import run, setup, write_index
from scipy_consumer import recorded_curve_fit
from workflow_consumer import affine, recorded_evaluate, recorded_prepare

print(sys.executable)
destination, store, x, y, p0 = setup(Path("artifacts/workflow"))

In [ ]:
handle = recorda.start(
    "prepare-fit-evaluate",
    path=destination / "workflow.jsonl",
    reference_adapters=store.adapters,
    gaps=["undeclared helpers and external work"],
)
assert handle.record.status == "incomplete"

In [ ]:
prepared = recorded_prepare(x, y)
assert prepared.kept_rows == (0, 1, 2, 3, 4, 5)
print("finite rows:", prepared.kept_rows)

In [ ]:
fit = recorded_curve_fit(
    affine,
    prepared.x,
    prepared.y,
    p0=p0,
    method="lm",
    maxfev=1000,
)
print("slope, intercept:", fit[0])

In [ ]:
evaluation = recorded_evaluate(prepared, fit)
print("residual sum of squares:", evaluation.rss)

In [ ]:
record = handle.stop()
write_index(store)
assert record.status == "succeeded"
assert len(record.operations) == 3

In [ ]:
report = inspect_workflow(destination)
assert report["scientific_check"]["status"] == "consistent"
assert len(report["dependency_links"]) == 4
print(report["scientific_check"])
print(report["dependency_links"])

In [ ]:
faults = run(Path("artifacts/workflow-cases"))["reports"]
assert faults["retry"]["session_status"] == "failed"
assert faults["interrupted"]["session_status"] == "incomplete"
assert faults["minimal"]["dependency_links"] == []
assert faults["missing_intermediate"]["session_status"] == "succeeded"
assert faults["restored"] == faults["full"]
print({name: value["scientific_check"]["status"] for name, value in faults.items()})